# Stage 6 — Place the tiles

**The question:** which squares of this slide should the encoder see?

`get_tiles` lays a regular grid over the tissue mask, scores every cell by how
much tissue it holds, keeps the best cells up to a cap, and reads those tiles.
It opens the slide file and nothing else, so it runs unchanged on test slides.

The cells after `get_tiles` check it: its output, where the tiles land, what
they look like, whether it repeats exactly, and how much of the annotated
cancer the tiles capture across a sweep of tissue floors, with padded and
trimmed edges. The label masks are used only in that sweep, to score the result,
never to place a tile.

This notebook writes no files. The tile selector and its settings live in
`src/tiling.py`; this notebook checks them and records why the settings are
what they are.

In [ ]:
"""Imports.

The first block makes the project's src/ folder importable. Locally it sits
one folder up from notebooks/. On Kaggle it arrives as an attached dataset, so
every attached dataset is checked for a src/ folder, and the notebook stops
with a clear message if there is none.

math       rounds panel counts up when laying out figure grids.
numpy      holds masks, scores and positions as arrays.
openslide  opens the label masks, in the sweep only.
pandas     loads the inventory and colour statistics, and holds the results.
matplotlib draws the sweep charts and tile grids.
tqdm       draws progress bars for the two sampling loops.
src.paths    where the data is, locally and on Kaggle.
src.tiling   the tile selector and its settings: the thing this notebook checks.
src.display  show_positions and show_tiles, the two pictures of the tiling.
"""

import glob
import os
import sys

SRC_ROOT = next((root for root in [os.path.abspath("..")] + sorted(glob.glob("/kaggle/input/*"))
                 if os.path.isfile(os.path.join(root, "src", "tiling.py"))), None)
if SRC_ROOT is None:
    raise FileNotFoundError("src/ not found; on Kaggle, attach the project's src folder as a dataset")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

import math

import numpy as np
import openslide
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from src import tiling
from src.paths import MASK_DIR, find_input, slide_path
from src.tiling import (MIN_FRAC, PAD_EDGES, READ_LEVEL, TILE_PER_SLIDE_CAP, TILE_SIZE,
                        get_tiles, read_smallest_level, rgba_to_rgb_white,
                        score_grid, select_cells, tissue_mask)
from src.display import show_positions, show_tiles

In [ ]:
"""Inputs and settings.

1. The two inputs: the Stage 1 inventory and the Stage 2 colour statistics.
   Both are used only to choose which slides to check, never by get_tiles.
2. SEED fixes which slides the random samples below pick.

The tile placement settings are not here: they live in src/tiling.py, next to
the code that uses them.
"""

INVENTORY_PATH = find_input("slide_inventory.parquet")
COLOUR_STATS_PATH = find_input("slide_colour_stats.parquet")

SEED = 0

## The tile selector

The code is in `src/tiling.py`. `get_tiles` does six things, in this order,
each in its own function:

1. read the slide's smallest pyramid level;
2. find the tissue on it, with Stage 5's detector;
3. lay a grid over it, with cells the size of one tile, and score every cell
   by the share of it that is tissue;
4. keep the best-scoring cells above the floor, up to the cap;
5. convert the kept cells to level-0 positions;
6. read those tiles from `READ_LEVEL`.

The module reads the slide file and nothing else: no masks, no CSVs, no
labels, no hospital. That is what lets it run on test slides. The first check
below makes sure it stays that way.

Three coordinate systems meet here. The mask is the smallest level, about 16
times smaller than full resolution. Tiles are read at level 1, 4 times smaller.
`read_region` takes its position in **level-0 pixels always**, whatever level
it reads, and its size in the pixels of the level it reads. Positions are
stored in level-0 pixels as `(x, y)`, never anything else.

In [ ]:
"""Check that src/tiling.py imports nothing but image libraries.

1. Read the module's source and list every module it imports.
2. Require that list to be within cv2, numpy, openslide and PIL. An import of
   pandas, of src.paths, or of anything else that could reach a label, a mask
   or a CSV stops the notebook here. That is how the rule "tiling reads the
   image file and nothing else" gets broken by accident: through an
   innocent-looking import.
3. Print the settings the module holds.
"""

import ast
import inspect

tree = ast.parse(inspect.getsource(tiling))
imported = ({alias.name.split(".")[0] for node in ast.walk(tree) if isinstance(node, ast.Import)
             for alias in node.names}
            | {node.module.split(".")[0] for node in ast.walk(tree) if isinstance(node, ast.ImportFrom)})
assert imported <= {"cv2", "numpy", "openslide", "PIL"}, f"tiling imports {sorted(imported)}"
print(f"src/tiling.py imports only: {sorted(imported)}")
print(f"tile {TILE_SIZE} px at level {READ_LEVEL}; floor {MIN_FRAC}; "
      f"{'padded' if PAD_EDGES else 'trimmed'} edges; cap {TILE_PER_SLIDE_CAP}")

## Checking the tile selector

Everything below uses `get_tiles` or its building blocks as they are defined
above. Nothing here changes how tiles are chosen.

In [ ]:
"""Load the inventory and attach each slide's colour statistics.

Used only to pick slides for the checks below and to look up their sizes and
grades; get_tiles never sees any of it.

1. Read the Stage 1 inventory and the Stage 2 colour statistics, and join them
   on slide_id. how="left" keeps every inventory slide.
2. area is each slide's level-0 area, width times height.
3. Print the table's shape and show the first rows.
"""

inventory = pd.read_parquet(INVENTORY_PATH).merge(pd.read_parquet(COLOUR_STATS_PATH), on="slide_id", how="left")
inventory["area"] = inventory["tiff_lvl0_dim"].apply(lambda d: int(d[0]) * int(d[1]))

print(inventory.shape)
inventory.head(3)

In [ ]:
"""Slides to look at, chosen once and reused by the checks below.

1. A repeatable random sample of 5 slides per hospital.
2. The awkward cases, where tile placement goes wrong first: the slide with no
   detectable tissue (Stage 5), the palest slide that has tissue, the slide
   with the most pen, and the smallest and largest slides.
"""

random_slides = {
    provider: inventory[inventory["provider"] == provider].sample(5, random_state=SEED)["slide_id"].tolist()
    for provider in ("radboud", "karolinska")
}
NO_TISSUE_SLIDE = "3790f55cad63053e956fb73027179707"
awkward_slides = {
    "no tissue": NO_TISSUE_SLIDE,
    "palest": inventory[inventory["slide_id"] != NO_TISSUE_SLIDE].nsmallest(1, "saturation_mean")["slide_id"].iloc[0],
    "most pen": inventory.nlargest(1, "ink_frac")["slide_id"].iloc[0],
    "smallest": inventory.nsmallest(1, "area")["slide_id"].iloc[0],
    "largest": inventory.nlargest(1, "area")["slide_id"].iloc[0],
}
for name, slide_id in awkward_slides.items():
    print(f"{name:<10} {slide_id}")

In [ ]:
"""Check what get_tiles returns, on the awkward slides.

For each slide:
1. Call get_tiles with the current settings.
2. Check the contract: one tile, one position and one fraction per tile;
   positions are 32-bit integers inside the slide's level-0 width and height,
   looked up in the inventory;
   fractions are 32-bit floats at or above the floor; every tile is a
   TILE_SIZE x TILE_SIZE RGB image.
3. Print the tile count and whether the contract holds.

The no-tissue slide should return three empty results of the right shapes,
not an error.
"""

for name, slide_id in awkward_slides.items():
    tiles, coords, fracs = get_tiles(slide_path(slide_id))
    width, height = inventory.loc[inventory["slide_id"] == slide_id, "tiff_lvl0_dim"].iloc[0]
    contract_ok = (
        len(tiles) == len(coords) == len(fracs)
        and coords.dtype == np.int32 and fracs.dtype == np.float32
        and all(t.size == (TILE_SIZE, TILE_SIZE) and t.mode == "RGB" for t in tiles)
        and bool((fracs >= MIN_FRAC).all())
        and bool((coords[:, 0] < width).all()) and bool((coords[:, 1] < height).all())
    )
    print(f"{name:<10} {slide_id[:8]}: {len(tiles):>3} tiles, shapes {coords.shape} {fracs.shape}, "
          f"contract ok: {contract_ok}")

## Where did the tiles land?

Drawing the chosen tiles back onto the slide is the check that catches a
coordinate mistake. If the conversion between the mask, level 1 and level 0 is
wrong, the squares drift off the tissue or bunch in a corner, and no number
shows it.

In [ ]:
"""Draw each slide's chosen tiles as outlined squares on its smallest level,
with show_positions from src/display.py: 5 random slides from each hospital,
then the awkward slides."""

for provider, slide_list in random_slides.items():
    show_positions(slide_list, suptitle=f"{provider} — chosen tiles")
show_positions(list(awkward_slides.values()), titles=list(awkward_slides.keys()),
               suptitle="awkward slides — chosen tiles")
plt.show()

## The tiles themselves

The positions can be right and the reads still wrong: a level mix-up gives real
image from the wrong place, and a dropped transparency channel gives black
edges. Looking at the tiles is the only way to see either.

In [ ]:
"""The first 24 tiles of a slide, in reading order, each captioned with its
tissue fraction, with show_tiles from src/display.py.

Shown for one random slide from each hospital, and for the slide with the most
pen, where ink should be visible in some tiles but must not have counted
towards their tissue fraction.
"""

for provider, slide_list in random_slides.items():
    show_tiles(slide_list[0], suptitle=f"{provider} — first 24 tiles")
show_tiles(awkward_slides["most pen"], suptitle="most pen — first 24 tiles")
plt.show()

## Is it repeatable?

At submission time the tiling runs live on slides nobody has seen, and its
features must match the ones the model trained on. If two runs on the same
slide can differ, that later check fails and the cause is hard to find.

In [ ]:
"""Call get_tiles twice on each of 5 slides and require identical output:
the same positions, the same tissue fractions, and the same pixels in every
tile. Pixels are compared too, because stable positions with unstable reads
would still break the later reproduction check."""

for slide_id in random_slides["radboud"][:3] + random_slides["karolinska"][:2]:
    tiles_a, coords_a, fracs_a = get_tiles(slide_path(slide_id))
    tiles_b, coords_b, fracs_b = get_tiles(slide_path(slide_id))
    assert np.array_equal(coords_a, coords_b), f"{slide_id}: positions differ"
    assert np.array_equal(fracs_a, fracs_b), f"{slide_id}: fractions differ"
    assert all(np.array_equal(np.asarray(a), np.asarray(b)) for a, b in zip(tiles_a, tiles_b)), \
        f"{slide_id}: pixels differ"
print("repeatable on 5 slides: positions, fractions and pixels identical")

## Sweep: tissue floor and edge handling, scored against the masks

How much of the annotated cancer ends up inside the chosen tiles, for a range of
tissue floors, with edges padded and with edges trimmed.

The two settings are not independent of the cap: a lower floor admits more
cells, and on a big slide the cap then decides which survive. So every
combination is measured as `get_tiles` would run it, with the cap in place.

This is the one place in the notebook that reads label masks. The masks score
where the tiles landed; they never choose where tiles go.

In [ ]:
"""Sweep every floor, padded and trimmed, over slides that have annotated cancer.

1. read_cancer_mask reads a slide's label mask at its smallest level, keeps the
   raw red channel (the class number; no compositing, since a class number
   blended with white would stop being one), and marks the cancer classes:
   3, 4 and 5 on Radboud, 2 on Karolinska.
2. cancer_captured paints every chosen cell onto a blank image the size of the
   mask and returns the share of cancer pixels that fall inside.
3. For each hospital, take a repeatable sample of SWEEP_SAMPLE slides that have
   masks, and skip slides whose mask holds no cancer.
4. For each slide, build the tissue mask once and score the grid once padded
   and once trimmed, the same way get_tiles does. Then, for every floor, choose
   cells with select_cells under the cap, and record the share of cancer
   captured and the number of tiles. A floor of 0 means any tissue at all.
5. Also record how much cancer the detector marks on its own, at pixel level.
6. Only numbers are kept, never images, so memory stays flat.
"""

SWEEP_FLOORS = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25]
SWEEP_SAMPLE = 80


def read_cancer_mask(slide_id, provider):
    """True where the label mask marks cancer, at the smallest pyramid level."""
    mask_slide = openslide.OpenSlide(os.path.join(MASK_DIR, f"{slide_id}_mask.tiff"))
    try:
        level = mask_slide.level_count - 1
        region = mask_slide.read_region((0, 0), level, mask_slide.level_dimensions[level])
    finally:
        mask_slide.close()
    labels = np.asarray(region)[..., 0]
    return ((labels >= 3) & (labels <= 5)) if provider == "radboud" else (labels == 2)


def cancer_captured(coords, cell_size, cancer):
    """Share of cancer pixels inside the chosen cells (coords are mask rows, cols)."""
    chosen = np.zeros_like(cancer)
    for row, col in coords:
        chosen[row:row + cell_size, col:col + cell_size] = True
    return (chosen & cancer).sum() / cancer.sum()


sweep_rows = []
with_mask = inventory[inventory["mask_exists"]]
for provider in ("radboud", "karolinska"):
    sample = with_mask[with_mask["provider"] == provider].sample(SWEEP_SAMPLE, random_state=2)
    for r in tqdm(list(sample.itertuples()), desc=f"sweep {provider}"):
        image, ds = read_smallest_level(slide_path(r.slide_id))
        mask = tissue_mask(np.asarray(rgba_to_rgb_white(image)))
        cancer = read_cancer_mask(r.slide_id, provider)
        if cancer.shape != mask.shape or cancer.sum() == 0:
            continue
        cell_size = int(round(TILE_SIZE * ds[READ_LEVEL] / ds[-1]))
        sweep_rows.append({"provider": provider, "edges": "detector", "floor": np.nan,
                           "captured": (mask & cancer).sum() / cancer.sum(), "tiles": np.nan})
        for edges, pad in (("padded", True), ("trimmed", False)):
            fractions = score_grid(mask, cell_size, pad)
            for floor in SWEEP_FLOORS:
                coords, _ = select_cells(fractions, floor if floor > 0 else 1e-9, TILE_PER_SLIDE_CAP)
                sweep_rows.append({"provider": provider, "edges": edges, "floor": floor,
                                   "captured": cancer_captured(coords, cell_size, cancer),
                                   "tiles": len(coords)})

sweep = pd.DataFrame(sweep_rows)
slides_scored = sweep[sweep["edges"] == "detector"]["provider"].value_counts().to_dict()
print(f"slides with annotated cancer: {slides_scored}")

In [ ]:
"""The sweep as a table.

For every edge setting and floor, per hospital: the median, 5th percentile and
worst share of cancer captured, the number of slides that lost more than half
their cancer, and the median and largest tile count. The detector row is the
share the tissue mask marks on its own, before any grid.
"""


def summarise(group):
    return pd.Series({
        "median": group["captured"].median(),
        "p5": group["captured"].quantile(0.05),
        "worst": group["captured"].min(),
        "under_half": int((group["captured"] < 0.5).sum()),
        "tiles_median": group["tiles"].median(),
        "tiles_max": group["tiles"].max(),
    })


summary = (sweep.fillna({"floor": -1})
                .groupby(["edges", "floor", "provider"])[["captured", "tiles"]]
                .apply(summarise)
                .round(3))
print(summary.to_string())

In [ ]:
"""The sweep as two charts.

Left: the worst slide's share of cancer captured, against the floor. The worst
slide is plotted rather than the median because missed cancer is the expensive
error and it happens on the worst slides. Right: the median tile count, the
cost of a lower floor. Colour is the hospital; a solid line is padded edges, a
dashed line trimmed. Karolinska's dashed line is hidden under its solid one:
trimming changes nothing there.
"""

colours = {"radboud": "#2a78d6", "karolinska": "#eb6834"}
styles = {"padded": "-", "trimmed": "--"}
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for (edges, provider), rows in summary.reset_index().query("edges != 'detector'").groupby(["edges", "provider"]):
    axes[0].plot(rows["floor"], rows["worst"], styles[edges], color=colours[provider],
                 marker="o", ms=4, label=f"{provider}, {edges}")
    axes[1].plot(rows["floor"], rows["tiles_median"], styles[edges], color=colours[provider],
                 marker="o", ms=4, label=f"{provider}, {edges}")
axes[0].set_xlabel("minimum tissue fraction (0 = any tissue)")
axes[0].set_ylabel("worst slide: share of cancer captured")
axes[1].set_xlabel("minimum tissue fraction (0 = any tissue)")
axes[1].set_ylabel("median tiles per slide")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, frameon=False)
fig.tight_layout()
plt.show()

## Look at the tiles a floor would add or drop

The sweep says how much cancer a floor loses; it cannot say whether the extra
tiles a lower floor admits are useful tissue or blank glass. Only the pictures
can. These are the tiles whose tissue fraction falls between two candidate
floors.

In [ ]:
"""Tiles whose tissue fraction lies between two floors, across several slides.

show_tiles_in_band:
1. For each slide, run get_tiles with the lower floor, so the band's tiles are
   included, and keep only tiles whose fraction is below the upper floor.
2. Stop once n tiles are collected.
3. Draw them, captioned with their fraction.

Shown for anything above 0% up to 5% (the tiles a lower floor would add), for
5% to 10% and for 10% to 25%, over the random slides from both hospitals.
"""


def show_tiles_in_band(low, high, slide_ids, n=32, ncols=8):
    """Up to n tiles with low <= tissue fraction < high."""
    band = []
    for slide_id in slide_ids:
        tiles, _, fracs = get_tiles(slide_path(slide_id), min_frac=low)
        band += [(tile, frac) for tile, frac in zip(tiles, fracs) if frac < high]
        if len(band) >= n:
            break
    band = band[:n]
    nrows = max(1, math.ceil(len(band) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 1.3, nrows * 1.45))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.set_axis_off()
    for ax, (tile, frac) in zip(axes, band):
        ax.imshow(tile)
        ax.set_title(f"{frac:.2f}", fontsize=6)
    fig.suptitle(f"tiles with tissue fraction {low:.0%} to {high:.0%} ({len(band)} shown)", fontsize=11)
    fig.tight_layout()
    return fig


band_slides = random_slides["radboud"] + random_slides["karolinska"]
show_tiles_in_band(1e-9, 0.05, band_slides)
show_tiles_in_band(0.05, 0.10, band_slides)
show_tiles_in_band(0.10, 0.25, band_slides)
plt.show()

## What the band grids show

- **Above 0% up to 5%.** Mostly glass, with a sliver of tissue or a dye-coated
  edge at one side. These are what a floor of zero would add.
- **5% to 10%.** Thin strips of real tissue along the edge of a fragment, often
  with a lot of glass beside them. None is blank. Losing these is what the
  sweep's drop from 5% to 10% measures: the worst Radboud slide falls from
  0.953 to 0.847 of its cancer.
- **10% to 25%.** Many are tissue edges covered in blue marking dye. The detector
  removes dye as ink, so these tiles hold more tissue than their fraction says,
  and a higher floor would drop them first.
- A few tiles carry some dye or pen alongside tissue. That is expected: ink is
  kept out of the score, never out of the tile.

## Tile counts and overhang, over a sample

**Tile count** is feedback on the settings: how many tiles a typical slide gets,
how often the cap binds, and which slides come out thin.

**Overhang** is how many tiles extend past the slide's edge. With padded edges
this is expected, and the white compositing in `rgba_to_rgb_white` fills the
part outside the slide; this measures how much that step is actually used.

In [ ]:
"""Run get_tiles over a sample and keep only numbers per slide.

1. Take a repeatable sample of COUNT_SAMPLE slides per hospital.
2. For each slide, run get_tiles with the current settings, and record the tile
   count, the median tissue fraction of the kept tiles, and how many tiles
   overhang: a tile at level-0 position (x, y) covers TILE_SIZE x the read
   level's downsample pixels, so it overhangs if that runs past the slide's
   width or height. Both come from the inventory.
3. Report tiles per slide per hospital, how many slides hit the cap, the
   thinnest slides, the share of tiles that overhang, and the median tile
   count by grade. If higher grades got systematically more tiles, a model
   could learn the number of tiles as a shortcut.
"""

COUNT_SAMPLE = 60

count_rows = []
for provider in ("radboud", "karolinska"):
    sample = inventory[inventory["provider"] == provider].sample(COUNT_SAMPLE, random_state=SEED)
    for r in tqdm(list(sample.itertuples()), desc=f"tiles {provider}"):
        width, height = r.tiff_lvl0_dim
        _, coords, fracs = get_tiles(slide_path(r.slide_id))
        span = TILE_SIZE * r.tiff_lvl_ds[READ_LEVEL]
        overhang = ((coords[:, 0] + span > width) | (coords[:, 1] + span > height)).sum() if len(coords) else 0
        count_rows.append({"slide_id": r.slide_id, "provider": provider, "isup_grade": r.isup_grade,
                           "tiles": len(coords), "overhang": int(overhang),
                           "median_frac": float(np.median(fracs)) if len(fracs) else 0.0})

counts = pd.DataFrame(count_rows)
print("tiles per slide")
print(counts.groupby("provider")["tiles"].describe(percentiles=[.05, .5, .95]).round(1).to_string())
print(f"\nslides at the cap of {TILE_PER_SLIDE_CAP}: {(counts['tiles'] >= TILE_PER_SLIDE_CAP).sum()} of {len(counts)}")
print("thinnest slides:")
print(counts.nsmallest(5, "tiles")[["slide_id", "provider", "isup_grade", "tiles"]].to_string(index=False))
print(f"\ntiles overhanging the slide edge: {counts['overhang'].sum()} of {counts['tiles'].sum()} "
      f"({100 * counts['overhang'].sum() / max(counts['tiles'].sum(), 1):.2f}%)")
print("\nmedian tiles per slide by grade:")
print(counts.groupby("isup_grade")["tiles"].median().to_string())

## Decide

The decision is recorded in `src/tiling.py`, as `MIN_FRAC`, `PAD_EDGES` and
`TILE_PER_SLIDE_CAP`, next to the code that uses them. To change it, edit the
module and re-run this notebook: every check above then uses the new values.
Changing it after features have been extracted means extracting them again.

**Floor 5%.** The tiles between 5% and 10% are real tissue, not glass.
Raising the floor to 10% drops them and lowers the worst slide from 0.980 to
0.916 of its cancer on Karolinska, and from 0.953 to 0.847 on Radboud. Going
down to any tissue at all adds about 10 tiles per slide (a median of 84 and 74
against 72 and 64.5), and the band grid shows what they are: mostly glass, with
a sliver of tissue or a dye-coated edge at one side. They lift the worst slide
from 0.980 to 0.993 on Karolinska and from 0.953 to 0.999 on Radboud; the 5th
percentile moves by under 2 points.

**Padded edges.** Trimming drops the partial last row and column. It changes
nothing on Karolinska and loses cancer on Radboud: at a 5% floor, the 5th
percentile falls from 0.993 to 0.968.

**Cap 144.** It binds on 1 slide in 120, so it is a guard against very large
slides, not a setting that shapes typical ones.

In [ ]:
"""The decision, as src/tiling.py holds it.

1. Print the settings get_tiles uses.
2. Print the sweep's result for exactly this combination, so the decision and
   its evidence sit side by side. If the floor is not one of the swept values,
   say so rather than print nothing.
"""

print("Tile placement settings (src/tiling.py)")
print(f"  tile size:        {TILE_SIZE} px at level {READ_LEVEL}")
print(f"  tissue floor:     {MIN_FRAC}")
print(f"  edges:            {'padded' if PAD_EDGES else 'trimmed'}")
print(f"  cap:              {TILE_PER_SLIDE_CAP} tiles per slide")

key = ("padded" if PAD_EDGES else "trimmed", MIN_FRAC)
if key in summary.droplevel("provider").index:
    print("\nsweep result for these settings:")
    print(summary.loc[key].to_string())
else:
    print(f"\nfloor {MIN_FRAC} is not in the sweep {SWEEP_FLOORS}; add it there before relying on it.")

## Result

- `get_tiles` reads only the slide file, returns the same tiles, positions and
  fractions on every run, and returns empty results rather than failing on a
  slide with no tissue.
- Chosen: tissue floor 5%, padded edges, cap 144.
- At that setting the tiles hold a median 99.4% of the annotated cancer on
  Karolinska and 99.7% on Radboud; the worst slides keep 98.0% and 95.3%.
- Median tiles per slide: 72 on Karolinska, 66 on Radboud. 0.35% of tiles
  overhang the slide edge.
- Median tile count varies by grade, from 57 at grade 5 to 77.5 at grade 4.
  If a model can read the number of tiles, that is a possible shortcut; worth
  checking once there is a model.

All of this is recorded in `FINDINGS.md` under Stage 6.